# 01 · Preprocessing (Stage 1: Data Preparation)
**Project:** Evaluating the Robustness and Resilience of Deep Learning Intrusion Detection Systems Against LLM-Generated Polymorphic Attack Traffic

This notebook implements **Section 8.1** of the project plan:

1. Download the CIC-IDS2017 `MachineLearningCSV` files straight into Google Drive (no manual upload).
2. Load **only** the 3 CSV files that contain our chosen classes (chunked, `float32`).
3. Keep `Benign` + 3 attack classes: **DoS Hulk**, **PortScan**, **Brute Force** (`FTP-Patator` + `SSH-Patator` merged).
4. Clean `Inf` / `-Inf` / `NaN`, impossible negative values, exact duplicates and label-conflicting duplicates, all **before** splitting (no leakage).
5. Drop identifier columns (Flow ID, IPs, ports of the source, Timestamp) when present.
6. Stratified **70 / 15 / 15** split → down-sample `Benign` in train/val only; the test set keeps the **realistic class mix**.
7. Fit `StandardScaler` on **train only**; transform val/test.
8. Define and save `mutable_features`, `fixed_features`, semantic feature groups and **min/max bounds** (global + per-class) from the training data for Stage 3.

---
## How the dataset download works (fully automatic)
Cell **4** tries these sources in order and stops at the first one that works. Only the 3 needed CSVs are extracted to Drive:

| Order | Source | What you need to do |
|---|---|---|
| 1 | Hugging Face mirror of the official archive (`bencorn/CICIDS2017`, `csvs/MachineLearningCSV.zip`, ~235 MB) | Nothing |
| 2 | Official CIC/UNB server (`MachineLearningCSV.zip`) | Nothing (URL may change over time) |
| 3 | Kaggle API (`chethuhn/network-intrusion-dataset`) | Put your `kaggle.json` (Kaggle → Settings → *Create New Token*) at `MyDrive/IDS_Polymorphic_Project/kaggle.json` |
| 4 | Manual | Upload the 3 CSVs below into `MyDrive/IDS_Polymorphic_Project/data/raw/` |

Files used (official names):
- `Tuesday-WorkingHours.pcap_ISCX.csv` → FTP-Patator, SSH-Patator, Benign
- `Wednesday-workingHours.pcap_ISCX.csv` → DoS Hulk, Benign
- `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv` → PortScan, Benign

The download runs **once**; afterwards the notebook detects the files on Drive and skips it.

**Runtime:** a CPU runtime is enough for this notebook (≈ 10–15 min the first time, mostly download + Drive writes; ≈ 3–5 min afterwards). Peak RAM ≈ 2–3 GB.

## 1 · Mount Google Drive and create the project folder structure

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/IDS_Polymorphic_Project'
    LOCAL_TMP = '/content/tmp_download'          # fast local Colab disk for the zip
else:
    BASE_DIR = os.path.abspath('./IDS_Polymorphic_Project')
    LOCAL_TMP = os.path.abspath('./tmp_download')

DIRS = {
    'base':      BASE_DIR,
    'raw':       os.path.join(BASE_DIR, 'data', 'raw'),
    'processed': os.path.join(BASE_DIR, 'data', 'processed'),
    'variants':  os.path.join(BASE_DIR, 'data', 'variants'),
    'models':    os.path.join(BASE_DIR, 'models'),
    'results':   os.path.join(BASE_DIR, 'results'),
    'prompts':   os.path.join(BASE_DIR, 'prompts'),
    'config':    os.path.join(BASE_DIR, 'config'),
    'logs':      os.path.join(BASE_DIR, 'logs'),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
os.makedirs(LOCAL_TMP, exist_ok=True)

for k, v in DIRS.items():
    print(f'{k:>10}: {v}')

## 2 · Imports and helpers

In [ ]:
import gc, re, json, time, shutil, zipfile, subprocess, platform, random
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import requests
import joblib
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=True)
    from tqdm.auto import tqdm

try:
    import psutil
    def mem_usage():
        rss = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        avail = psutil.virtual_memory().available / 1024**3
        return f'process RAM {rss:.2f} GB | available {avail:.2f} GB'
except ImportError:
    def mem_usage():
        return 'psutil not available'

def df_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
print('python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| sklearn', sklearn.__version__, '| matplotlib', matplotlib.__version__)
print(mem_usage())

## 3 · Experiment configuration
Everything that later notebooks need (classes, seeds, paths) is saved to `config/project_config.json` at the end of this notebook.

**Class choice.** `FTP-Patator` and `SSH-Patator` are merged into one `BruteForce` class (same attack family, repeated login attempts; each alone is small after de-duplication). To keep them separate, map them to two different names in `LABEL_MAP` and add both names to `CLASS_NAMES`. To use DoS GoldenEye instead of DoS Hulk, change the `LABEL_MAP` key to `'DoS GoldenEye'` (it is in the same Wednesday file).

**Balancing.** Each attack class in train is capped at `ATTACK_CAP_TRAIN` rows (keeps the LSTM affordable on free Colab across 3 seeds × 2 retraining rounds), and `Benign` is down-sampled to `BENIGN_TO_ATTACK_RATIO` × (number of attack rows). Validation uses the same rule (cap scaled by 15/70). The **test set is never re-balanced**: it keeps the cleaned dataset's real class proportions (optionally stratified-subsampled to `TEST_MAX_ROWS` rows, which preserves those proportions).

In [ ]:
# ---------------- reproducibility ----------------
DATA_SEED   = 42                 # seed for all data splits / sampling in this notebook
MODEL_SEEDS = [42, 102, 2025]    # seeds used for every model run in notebooks 02-05
random.seed(DATA_SEED); np.random.seed(DATA_SEED)

# ---------------- files & classes ----------------
REQUIRED_FILES = [
    'Tuesday-WorkingHours.pcap_ISCX.csv',                    # FTP-Patator, SSH-Patator
    'Wednesday-workingHours.pcap_ISCX.csv',                  # DoS Hulk (+ other DoS we ignore)
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',  # PortScan
]

# raw CIC-IDS2017 label  ->  project class name  (matching is case-insensitive)
LABEL_MAP = {
    'BENIGN':      'Benign',
    'DoS Hulk':    'DoS_Hulk',
    'PortScan':    'PortScan',
    'FTP-Patator': 'BruteForce',
    'SSH-Patator': 'BruteForce',
}
CLASS_NAMES  = ['Benign', 'DoS_Hulk', 'PortScan', 'BruteForce']   # index == class id
BENIGN_CLASS = 'Benign'
ATTACK_CLASSES = [c for c in CLASS_NAMES if c != BENIGN_CLASS]

assert CLASS_NAMES[0] == BENIGN_CLASS, 'Benign must be class 0 (used for evasion-rate computation).'
assert set(LABEL_MAP.values()) == set(CLASS_NAMES), 'LABEL_MAP values and CLASS_NAMES must match.'
LABEL_MAP_UPPER = {k.strip().upper(): v for k, v in LABEL_MAP.items()}

# ---------------- split & balancing ----------------
TEST_SIZE  = 0.15
VAL_SIZE   = 0.15
TRAIN_SIZE = 1.0 - TEST_SIZE - VAL_SIZE          # 0.70
ATTACK_CAP_TRAIN       = 40_000                  # max rows per attack class in TRAIN (None = no cap)
ATTACK_CAP_VAL         = int(ATTACK_CAP_TRAIN * VAL_SIZE / TRAIN_SIZE) if ATTACK_CAP_TRAIN else None
BENIGN_TO_ATTACK_RATIO = 1.0                     # benign rows = ratio x total attack rows (train & val)
TEST_MAX_ROWS          = 200_000                 # stratified cap on the realistic test set (None = keep all)

# ---------------- columns ----------------
# identifier / environment-specific columns -> dropped if present (MachineLearningCSV already omits most)
IDENTIFIER_COLUMNS = ['Flow ID', 'Source IP', 'Src IP', 'Destination IP', 'Dst IP',
                      'Source Port', 'Src Port', 'Timestamp', 'SimillarHTTP', 'Unnamed: 0']
# known exact duplicate column in the CIC-IDS2017 CSVs
DUPLICATE_COLUMNS = ['Fwd Header Length.1']

CSV_CHUNKSIZE = 250_000

# ---------------- downloads ----------------
ZIP_SOURCES = [
    ('Hugging Face mirror (bencorn/CICIDS2017)',
     'https://huggingface.co/datasets/bencorn/CICIDS2017/resolve/main/csvs/MachineLearningCSV.zip'),
    ('CIC/UNB official server',
     'http://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip'),
]
KAGGLE_DATASET = 'chethuhn/network-intrusion-dataset'
KAGGLE_JSON_PATH = os.path.join(DIRS['base'], 'kaggle.json')

print('Train/Val/Test =', TRAIN_SIZE, VAL_SIZE, TEST_SIZE)
print('Attack cap train/val =', ATTACK_CAP_TRAIN, '/', ATTACK_CAP_VAL, '| benign ratio =', BENIGN_TO_ATTACK_RATIO)

## 4 · Automated dataset download into Google Drive
Skipped automatically if the 3 CSVs are already in `data/raw/`.

In [ ]:
def _required_lower():
    return {f.lower(): f for f in REQUIRED_FILES}

def raw_files_present():
    missing = []
    for f in REQUIRED_FILES:
        p = os.path.join(DIRS['raw'], f)
        if not (os.path.exists(p) and os.path.getsize(p) > 1024):
            missing.append(f)
    return len(missing) == 0, missing

def download_file(url, dest, timeout=60):
    with requests.get(url, stream=True, timeout=timeout, allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest, 'wb') as fh, tqdm(total=total, unit='B', unit_scale=True,
                                          desc=os.path.basename(dest)) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    fh.write(chunk)
                    bar.update(len(chunk))
    return dest

def extract_required_from_zip(zip_path):
    # extract ONLY the needed CSVs (whatever folder they sit in inside the zip) into data/raw
    req = _required_lower()
    found = 0
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            base = os.path.basename(info.filename)
            if base.lower() in req:
                dst = os.path.join(DIRS['raw'], req[base.lower()])
                print(f'  extracting {info.filename}  ->  {dst}')
                with zf.open(info) as src, open(dst, 'wb') as out:
                    shutil.copyfileobj(src, out, length=16 * 1024 * 1024)
                found += 1
    return found

def copy_required_from_dir(search_dir):
    req = _required_lower()
    found = 0
    for root, _, files in os.walk(search_dir):
        for f in files:
            if f.lower() in req:
                dst = os.path.join(DIRS['raw'], req[f.lower()])
                print(f'  copying {os.path.join(root, f)}  ->  {dst}')
                shutil.copy2(os.path.join(root, f), dst)
                found += 1
    return found

def try_zip_sources():
    for name, url in ZIP_SOURCES:
        zip_path = os.path.join(LOCAL_TMP, 'MachineLearningCSV.zip')
        print(f'\nTrying source: {name}\n  {url}')
        try:
            download_file(url, zip_path)
            if not zipfile.is_zipfile(zip_path):
                raise ValueError('downloaded file is not a valid zip archive')
            n = extract_required_from_zip(zip_path)
            print(f'  extracted {n}/{len(REQUIRED_FILES)} required files')
        except Exception as e:
            print(f'  FAILED: {type(e).__name__}: {e}')
        finally:
            if os.path.exists(zip_path):
                os.remove(zip_path)
        ok, _ = raw_files_present()
        if ok:
            return True
    return False

def try_kaggle():
    print(f'\nTrying source: Kaggle API ({KAGGLE_DATASET})')
    if not os.path.exists(KAGGLE_JSON_PATH):
        print(f'  kaggle.json not found at {KAGGLE_JSON_PATH} -> skipping Kaggle.')
        return False
    try:
        kdir = os.path.expanduser('~/.kaggle')
        os.makedirs(kdir, exist_ok=True)
        shutil.copy(KAGGLE_JSON_PATH, os.path.join(kdir, 'kaggle.json'))
        os.chmod(os.path.join(kdir, 'kaggle.json'), 0o600)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
        kaggle_dir = os.path.join(LOCAL_TMP, 'kaggle')
        os.makedirs(kaggle_dir, exist_ok=True)
        subprocess.run(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
                        '-p', kaggle_dir, '--unzip'], check=True)
        n = copy_required_from_dir(kaggle_dir)
        print(f'  copied {n}/{len(REQUIRED_FILES)} required files')
        shutil.rmtree(kaggle_dir, ignore_errors=True)
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}')
    return raw_files_present()[0]

ok, missing = raw_files_present()
if ok:
    print('All required CSV files already in Drive -> download skipped.')
else:
    print('Missing files:', missing)
    t0 = time.time()
    ok = try_zip_sources() or try_kaggle()
    print(f'\nDownload stage finished in {time.time() - t0:.0f}s')

ok, missing = raw_files_present()
if not ok:
    raise FileNotFoundError(
        'Could not obtain: ' + ', '.join(missing) +
        f"\nManual fallback: download MachineLearningCSV.zip from https://www.unb.ca/cic/datasets/ids-2017.html, "
        f"and upload the files above into: {DIRS['raw']}  -- then re-run this cell.")

for f in REQUIRED_FILES:
    p = os.path.join(DIRS['raw'], f)
    print(f'{f:<58} {os.path.getsize(p) / 1024**2:8.1f} MB')

## 5 · Feature-role rules (mutable vs fixed)
Defined **before** loading so the cleaning step can enforce non-negativity on behavioural (mutable) features.

- **Fixed** (identity of the attack; never changed by the LLM): destination port, protocol (if present), all TCP flag counts / PSH / URG flags, initial TCP window bytes, minimum segment size, and bulk-transfer fields.
- **Mutable** (statistical profile the LLM may change): duration, forward/backward packet counts, packet & segment length stats, total bytes, bytes/s & packets/s, IAT stats, active/idle times, sub-flow counts, header lengths, down/up ratio.

Header lengths, sub-flow counts and segment sizes are mathematically tied to packet counts/sizes, so they are mutable too — otherwise changing the counts would create inconsistent rows. Any column matching neither rule is treated as **fixed** (conservative) and printed as a warning.

In [ ]:
FIXED_PATTERN   = re.compile(r'(port|protocol|flag|init_win|min_seg|bulk)', re.I)
MUTABLE_PATTERN = re.compile(r'(duration|packet|pkt|length|bytes|iat|active|idle|subflow|header|ratio|size|segment|act_data)', re.I)

def feature_role(col):
    if FIXED_PATTERN.search(col):
        return 'fixed'
    if MUTABLE_PATTERN.search(col):
        return 'mutable'
    return 'fixed_unmatched'

SEMANTIC_GROUPS = {   # used by notebook 03 for logical-consistency checks
    'duration':         re.compile(r'^flow duration$', re.I),
    'packet_counts':    re.compile(r'(total (fwd|backward) packets|subflow (fwd|bwd) packets|act_data_pkt_fwd)', re.I),
    'byte_totals':      re.compile(r'(total length of (fwd|bwd) packets|subflow (fwd|bwd) bytes)', re.I),
    'packet_len_stats': re.compile(r'((fwd|bwd) packet length|^min packet length$|^max packet length$|packet length (mean|std|variance)|average packet size|avg (fwd|bwd) segment size)', re.I),
    'rates':            re.compile(r'(bytes/s|packets/s)$', re.I),
    'iat':              re.compile(r'iat', re.I),
    'active_idle':      re.compile(r'^(active|idle) ', re.I),
    'header':           re.compile(r'header length', re.I),
}
print('Example roles:', {c: feature_role(c) for c in
      ['Destination Port', 'Flow Duration', 'SYN Flag Count', 'Flow Bytes/s', 'Init_Win_bytes_forward', 'Fwd IAT Mean']})

## 6 · Memory-efficient chunked loading
Each CSV is read in chunks; for every chunk we strip column names, keep only the selected labels, drop identifier columns, coerce every feature to numeric and **downcast to `float32`**. Labels become a `category`.

In [ ]:
def load_filtered_csv(path):
    parts, n_read, raw_label_counts = [], 0, {}
    reader = pd.read_csv(path, chunksize=CSV_CHUNKSIZE, low_memory=False,
                         encoding='latin-1', skipinitialspace=True)
    for chunk in reader:
        n_read += len(chunk)
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.loc[:, ~chunk.columns.duplicated()]
        labels = chunk['Label'].astype(str).str.strip()
        for k, v in labels.value_counts().items():
            raw_label_counts[k] = raw_label_counts.get(k, 0) + int(v)
        mapped = labels.str.upper().map(LABEL_MAP_UPPER)
        keep = mapped.notna().to_numpy()
        chunk = chunk.loc[keep].copy()
        chunk['Label'] = mapped[keep].to_numpy()

        drop_cols = [c for c in IDENTIFIER_COLUMNS + DUPLICATE_COLUMNS if c in chunk.columns]
        chunk = chunk.drop(columns=drop_cols)
        feat_cols = [c for c in chunk.columns if c != 'Label']
        num = chunk[feat_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
        num['Label'] = chunk['Label'].to_numpy()
        parts.append(num)
        del chunk, num, labels, mapped
        gc.collect()
    df = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    return df, n_read, raw_label_counts

frames, load_report = [], {}
for f in REQUIRED_FILES:
    t0 = time.time()
    part, n_read, raw_counts = load_filtered_csv(os.path.join(DIRS['raw'], f))
    load_report[f] = {'rows_read': n_read, 'rows_kept': len(part), 'raw_label_counts': raw_counts}
    print(f'{f}: read {n_read:,} rows, kept {len(part):,} ({time.time() - t0:.0f}s) | {mem_usage()}')
    print('   raw labels:', raw_counts)
    frames.append(part)
    del part; gc.collect()

# align columns across files (they are identical for the official CSVs)
common_cols = [c for c in frames[0].columns if all(c in fr.columns for fr in frames)]
df = pd.concat([fr[common_cols] for fr in frames], ignore_index=True)
del frames; gc.collect()
df['Label'] = pd.Categorical(df['Label'], categories=CLASS_NAMES)

FEATURES_ALL = [c for c in df.columns if c != 'Label']
print(f'\nCombined: {df.shape[0]:,} rows x {len(FEATURES_ALL)} features | {df_mb(df):.0f} MB | {mem_usage()}')
print(df['Label'].value_counts().reindex(CLASS_NAMES))

## 7 · Cleaning (all done **before** splitting)
1. Rows with `Inf`, `-Inf` or `NaN` in any feature → removed.
2. Rows with negative values in **mutable** features (counts, lengths, durations, rates, IATs cannot be negative — known CICFlowMeter artefacts) → removed. Fixed features such as `Init_Win_bytes_* = -1` are legitimate ("not set") and are kept.
3. Exact duplicate rows → removed (keep first).
4. Rows whose **feature vector** appears with **different labels** (label conflicts, Engelen et al. 2021) → all copies removed.

After step 3+4 every feature vector is unique, so no identical flow can appear in both train and test.

In [ ]:
clean_log = {'initial_rows': int(len(df)), 'initial_counts': df['Label'].value_counts().reindex(CLASS_NAMES).astype(int).to_dict()}
X = df[FEATURES_ALL].to_numpy()

# 1) non-finite values
n_inf = int(np.isinf(X).any(axis=1).sum())
n_nan = int(np.isnan(X).any(axis=1).sum())
finite_mask = np.isfinite(X).all(axis=1)
del X; gc.collect()
df = df.loc[finite_mask].reset_index(drop=True)
clean_log['rows_with_inf'] = n_inf
clean_log['rows_with_nan'] = n_nan
clean_log['removed_nonfinite'] = int((~finite_mask).sum())
print(f'Rows with Inf: {n_inf:,} | rows with NaN: {n_nan:,} | removed (non-finite): {clean_log["removed_nonfinite"]:,}')

# 2) negative values in behavioural (mutable) features
mutable_candidates = [c for c in FEATURES_ALL if feature_role(c) == 'mutable']
neg_mask = (df[mutable_candidates].to_numpy() < 0).any(axis=1)
neg_by_col = (df[mutable_candidates] < 0).sum()
neg_by_col = neg_by_col[neg_by_col > 0].sort_values(ascending=False)
clean_log['removed_negative'] = int(neg_mask.sum())
clean_log['negative_by_column'] = {k: int(v) for k, v in neg_by_col.items()}
df = df.loc[~neg_mask].reset_index(drop=True)
print(f'Removed rows with negative mutable values: {clean_log["removed_negative"]:,}')
if len(neg_by_col):
    print(neg_by_col.head(10).to_string())

# 3) exact duplicates (features + label)
dup_mask = df.duplicated(keep='first').to_numpy()
clean_log['removed_exact_duplicates'] = int(dup_mask.sum())
dup_by_class = df.loc[dup_mask, 'Label'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict()
clean_log['exact_duplicates_by_class'] = dup_by_class
df = df.loc[~dup_mask].reset_index(drop=True)
print(f'Removed exact duplicates: {clean_log["removed_exact_duplicates"]:,}  by class: {dup_by_class}')

# 4) same features, different labels
conflict_mask = df.duplicated(subset=FEATURES_ALL, keep=False).to_numpy()
clean_log['removed_label_conflicts'] = int(conflict_mask.sum())
df = df.loc[~conflict_mask].reset_index(drop=True)
print(f'Removed label-conflicting rows: {clean_log["removed_label_conflicts"]:,}')

clean_log['final_rows'] = int(len(df))
clean_log['final_counts'] = df['Label'].value_counts().reindex(CLASS_NAMES).astype(int).to_dict()
del finite_mask, neg_mask, dup_mask, conflict_mask; gc.collect()

summary = pd.DataFrame({'before': clean_log['initial_counts'], 'after': clean_log['final_counts']})
summary['removed_%'] = (100 * (1 - summary['after'] / summary['before'])).round(2)
print(f'\nRows: {clean_log["initial_rows"]:,} -> {clean_log["final_rows"]:,} | {df_mb(df):.0f} MB | {mem_usage()}')
print(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
xs = np.arange(len(CLASS_NAMES)); w = 0.38
ax.bar(xs - w/2, summary['before'], w, label='before cleaning', color='#9aa5b1')
ax.bar(xs + w/2, summary['after'],  w, label='after cleaning',  color='#2f6db5')
for i, (b, a) in enumerate(zip(summary['before'], summary['after'])):
    ax.text(i + w/2, a, f'{a:,}', ha='center', va='bottom', fontsize=8)
ax.set_yscale('log'); ax.set_xticks(xs); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('flows (log scale)'); ax.set_title('CIC-IDS2017 selected classes: before vs after cleaning')
ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '01_class_distribution_cleaning.png'), dpi=150)
plt.show()

## 8 · Stratified 70 / 15 / 15 split, then balancing (train/val only)
1. Stratified split of the cleaned data: 15 % test vs 85 % rest; then the rest → 70 % train / 15 % val (of the total).
2. **Test:** realistic mix kept (stratified subsample to `TEST_MAX_ROWS` preserves proportions).
3. **Train / Val:** cap each attack class, then down-sample `Benign` to `BENIGN_TO_ATTACK_RATIO` × attack rows.

Balancing after splitting means no sampling decision can leak information between splits.

In [ ]:
y_all = df['Label'].cat.codes.to_numpy()
idx_all = np.arange(len(df))

idx_rest, idx_test = train_test_split(idx_all, test_size=TEST_SIZE, stratify=y_all, random_state=DATA_SEED)
idx_train, idx_val = train_test_split(idx_rest, test_size=VAL_SIZE / (1 - TEST_SIZE),
                                      stratify=y_all[idx_rest], random_state=DATA_SEED)

split_report = {'before_balancing': {}}
for name, idx in [('train', idx_train), ('val', idx_val), ('test', idx_test)]:
    split_report['before_balancing'][name] = pd.Series(y_all[idx]).map(dict(enumerate(CLASS_NAMES))) \
        .value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict()
print('Before balancing:'); print(pd.DataFrame(split_report['before_balancing']))

if TEST_MAX_ROWS is not None and len(idx_test) > TEST_MAX_ROWS:
    idx_test, _ = train_test_split(idx_test, train_size=TEST_MAX_ROWS, stratify=y_all[idx_test],
                                   random_state=DATA_SEED)

def balance_split(frame, attack_cap, benign_ratio, seed):
    parts, attack_total = [], 0
    for cls in ATTACK_CLASSES:
        sub = frame[frame['Label'] == cls]
        if attack_cap is not None and len(sub) > attack_cap:
            sub = sub.sample(n=attack_cap, random_state=seed)
        parts.append(sub); attack_total += len(sub)
    ben = frame[frame['Label'] == BENIGN_CLASS]
    n_ben = min(len(ben), int(round(benign_ratio * attack_total)))
    parts.append(ben.sample(n=n_ben, random_state=seed))
    out = pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    out['Label'] = pd.Categorical(out['Label'].astype(str), categories=CLASS_NAMES)
    return out

train_df = balance_split(df.iloc[idx_train], ATTACK_CAP_TRAIN, BENIGN_TO_ATTACK_RATIO, DATA_SEED)
val_df   = balance_split(df.iloc[idx_val],   ATTACK_CAP_VAL,   BENIGN_TO_ATTACK_RATIO, DATA_SEED)
test_df  = df.iloc[idx_test].sample(frac=1.0, random_state=DATA_SEED).reset_index(drop=True)
test_df['Label'] = pd.Categorical(test_df['Label'].astype(str), categories=CLASS_NAMES)

realistic_mix = (df['Label'].value_counts(normalize=True).reindex(CLASS_NAMES) * 100).round(3).to_dict()
del df, y_all, idx_all, idx_rest; gc.collect()

split_report['after_balancing'] = {n: d['Label'].value_counts().reindex(CLASS_NAMES).astype(int).to_dict()
                                   for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]}
split_report['realistic_mix_percent_full_clean_data'] = realistic_mix
after = pd.DataFrame(split_report['after_balancing'])
after.loc['TOTAL'] = after.sum()
print('\nAfter balancing (test keeps the realistic mix):'); print(after)
print('\nTest-set class share (%):', (test_df['Label'].value_counts(normalize=True).reindex(CLASS_NAMES) * 100).round(3).to_dict())
print('Full clean data share (%):', realistic_mix)
print(mem_usage())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['#2f6db5', '#d1495b', '#edae49', '#00798c', '#66a182', '#8d6a9f']
for ax, (name, d) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test (realistic)', test_df)]):
    counts = d['Label'].value_counts().reindex(CLASS_NAMES)
    ax.bar(CLASS_NAMES, counts, color=colors[:len(CLASS_NAMES)])
    for i, v in enumerate(counts):
        ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'{name}  (n={len(d):,})'); ax.tick_params(axis='x', rotation=20)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('flows')
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '01_split_composition.png'), dpi=150)
plt.show()

## 9 · Final feature set + `mutable_features` / `fixed_features`
Computed on the **training split only**:
- constant columns (zero variance in train) are dropped everywhere;
- columns that are an exact copy of an earlier column in train are dropped everywhere (e.g., sub-flow counts that equal total counts in CIC-IDS2017).

In [ ]:
nunique = train_df[FEATURES_ALL].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
candidate = [c for c in FEATURES_ALL if c not in constant_cols]

identical_cols, kept, seen_hash = {}, [], {}
for c in candidate:
    h = int(pd.util.hash_pandas_object(train_df[c], index=False).sum())
    match = None
    for other in seen_hash.get(h, []):
        if np.array_equal(train_df[c].to_numpy(), train_df[other].to_numpy()):
            match = other; break
    if match is not None:
        identical_cols[c] = match
    else:
        kept.append(c); seen_hash.setdefault(h, []).append(c)

FEATURES = kept
mutable_features = [c for c in FEATURES if feature_role(c) == 'mutable']
fixed_features   = [c for c in FEATURES if feature_role(c) != 'mutable']
unmatched        = [c for c in FEATURES if feature_role(c) == 'fixed_unmatched']
semantic_groups  = {g: [c for c in mutable_features if p.search(c)] for g, p in SEMANTIC_GROUPS.items()}

print(f'Dropped constant columns ({len(constant_cols)}): {constant_cols}')
print(f'Dropped identical-copy columns ({len(identical_cols)}): {identical_cols}')
print(f'\nFinal feature count: {len(FEATURES)}')
print(f'\nMUTABLE ({len(mutable_features)}):'); print(mutable_features)
print(f'\nFIXED ({len(fixed_features)}):'); print(fixed_features)
if unmatched:
    print('\nWARNING - columns matched no rule and are treated as FIXED:', unmatched)
print('\nSemantic groups (for Stage-3 consistency rules):')
for g, cols in semantic_groups.items():
    print(f'  {g:<17} {cols}')
assert len(set(mutable_features) & set(fixed_features)) == 0
assert set(mutable_features) | set(fixed_features) == set(FEATURES)

drop_final = constant_cols + list(identical_cols.keys())
for d in (train_df, val_df, test_df):
    d.drop(columns=drop_final, inplace=True)
gc.collect()

## 10 · Feature bounds from the training data (raw, unscaled)
Used by Stage 3 for range clipping and by the distribution sanity check:
- **global** min / max / 1st & 99th percentile / mean / std over the whole training set;
- **per class** statistics (attack classes are what the LLM rewrites).

In [ ]:
def describe_block(frame):
    arr = frame[FEATURES].to_numpy(dtype=np.float64)
    q = np.percentile(arr, [1, 50, 99], axis=0)
    return {f: {'min': float(arr[:, j].min()), 'max': float(arr[:, j].max()),
                'p01': float(q[0, j]), 'median': float(q[1, j]), 'p99': float(q[2, j]),
                'mean': float(arr[:, j].mean()), 'std': float(arr[:, j].std())}
            for j, f in enumerate(FEATURES)}

feature_bounds = {'global': describe_block(train_df),
                  'per_class': {c: describe_block(train_df[train_df['Label'] == c]) for c in CLASS_NAMES}}

bounds_table = pd.DataFrame(feature_bounds['global']).T[['min', 'p01', 'median', 'p99', 'max']]
bounds_table['role'] = ['mutable' if f in mutable_features else 'fixed' for f in bounds_table.index]
bounds_table.to_csv(os.path.join(DIRS['results'], '01_feature_bounds_train.csv'))
print(bounds_table.head(25))

# per-class medians of the mutable features (reference profile for Stage 3 prompts)
profile = train_df.groupby('Label', observed=True)[mutable_features].median().T
profile.to_csv(os.path.join(DIRS['results'], '01_mutable_feature_class_medians.csv'))
print('\nPer-class medians (first 12 mutable features):')
print(profile.head(12))

In [ ]:
key_feats = [f for f in ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                         'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean',
                         'Packet Length Mean', 'Fwd Packet Length Max'] if f in mutable_features][:8]
if len(key_feats) < 4:
    key_feats = mutable_features[:8]
ncols = 4; nrows = int(np.ceil(len(key_feats) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.6 * nrows))
axes = np.array(axes).reshape(-1)
for ax, f in zip(axes, key_feats):
    data = [np.log1p(train_df.loc[train_df['Label'] == c, f].to_numpy()) for c in CLASS_NAMES]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(CLASS_NAMES) + 1)); ax.set_xticklabels(CLASS_NAMES, rotation=20, fontsize=8)
    ax.set_title(f'log1p({f})', fontsize=9); ax.spines[['top', 'right']].set_visible(False)
for ax in axes[len(key_feats):]:
    ax.axis('off')
plt.suptitle('Training set: key mutable features per class (outliers hidden)')
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '01_mutable_feature_profiles.png'), dpi=150)
plt.show()

## 11 · Scaling (fit on train only) and saving all artefacts to Drive
Saved to `data/processed/`:
- `train_raw.parquet`, `val_raw.parquet`, `test_raw.parquet` — **unscaled** features + `Label` + `label_id` (Stage 3 samples real attack rows from `train_raw`).
- `X_train.npy`, `X_val.npy`, `X_test.npy` (`float32`, scaled) and `y_train.npy`, `y_val.npy`, `y_test.npy` (`int64`).
- `scaler.joblib` — the fitted `StandardScaler` (re-used to scale LLM variants in notebooks 04/05).

In [ ]:
label_to_id = {c: i for i, c in enumerate(CLASS_NAMES)}
for d in (train_df, val_df, test_df):
    d['label_id'] = d['Label'].map(label_to_id).astype(np.int64)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)
X_val   = scaler.transform(val_df[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)
X_test  = scaler.transform(test_df[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)
y_train = train_df['label_id'].to_numpy(); y_val = val_df['label_id'].to_numpy(); y_test = test_df['label_id'].to_numpy()

for n, a in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test)]:
    assert np.isfinite(a).all(), f'{n} contains non-finite values after scaling'
print('Train scaled mean (should be ~0):', float(np.abs(X_train.mean(axis=0)).max()).__round__(5),
      '| std (should be ~1):', float(np.median(X_train.std(axis=0))).__round__(4))
print('Val/Test scaled abs-max:', float(np.abs(X_val).max()), float(np.abs(X_test).max()))

P = DIRS['processed']
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = d[FEATURES + ['Label', 'label_id']].copy()
    out['Label'] = out['Label'].astype(str)
    out.to_parquet(os.path.join(P, f'{name}_raw.parquet'), index=False)
    del out
for name, arr in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
                  ('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    np.save(os.path.join(P, f'{name}.npy'), arr)
joblib.dump(scaler, os.path.join(P, 'scaler.joblib'))

class_weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_train)
class_weights = {int(i): float(w) for i, w in enumerate(class_weights)}
print('Balanced class weights (optional for notebook 02):', class_weights)
print('Saved arrays & parquet files to', P, '|', mem_usage())

## 12 · Save configuration, feature definitions and the preprocessing report
- `config/project_config.json` — paths, classes, seeds (read by every later notebook).
- `config/feature_config.json` — `feature_names`, `mutable_features`, `fixed_features`, `semantic_groups`, dropped columns, **bounds**.
- `results/01_preprocessing_report.json` — every count needed for the report's data section.

In [ ]:
project_config = {
    'project': 'Robustness of DL IDS against LLM-generated polymorphic attack traffic',
    'created': datetime.now().isoformat(timespec='seconds'),
    'dirs': DIRS,
    'dataset': 'CIC-IDS2017 MachineLearningCSV',
    'raw_files': REQUIRED_FILES,
    'label_map': LABEL_MAP,
    'class_names': CLASS_NAMES,
    'benign_class': BENIGN_CLASS,
    'benign_id': label_to_id[BENIGN_CLASS],
    'attack_classes': ATTACK_CLASSES,
    'label_to_id': label_to_id,
    'data_seed': DATA_SEED,
    'model_seeds': MODEL_SEEDS,
    'split': {'train': TRAIN_SIZE, 'val': VAL_SIZE, 'test': TEST_SIZE},
    'balancing': {'attack_cap_train': ATTACK_CAP_TRAIN, 'attack_cap_val': ATTACK_CAP_VAL,
                  'benign_to_attack_ratio': BENIGN_TO_ATTACK_RATIO, 'test_max_rows': TEST_MAX_ROWS},
    'class_weights_balanced': class_weights,
    'n_features': len(FEATURES),
    'processed_files': {
        'train_raw': 'train_raw.parquet', 'val_raw': 'val_raw.parquet', 'test_raw': 'test_raw.parquet',
        'X_train': 'X_train.npy', 'X_val': 'X_val.npy', 'X_test': 'X_test.npy',
        'y_train': 'y_train.npy', 'y_val': 'y_val.npy', 'y_test': 'y_test.npy',
        'scaler': 'scaler.joblib'},
}
feature_config = {
    'feature_names': FEATURES,
    'mutable_features': mutable_features,
    'fixed_features': fixed_features,
    'unmatched_treated_as_fixed': unmatched,
    'semantic_groups': semantic_groups,
    'dropped_identifier_columns': IDENTIFIER_COLUMNS + DUPLICATE_COLUMNS,
    'dropped_constant_columns': constant_cols,
    'dropped_identical_columns': identical_cols,
    'feature_bounds_train': feature_bounds,
    'role_rules': {'fixed_regex': FIXED_PATTERN.pattern, 'mutable_regex': MUTABLE_PATTERN.pattern},
}
report = {
    'load': load_report,
    'cleaning': clean_log,
    'splits': split_report,
    'final_shapes': {'X_train': list(X_train.shape), 'X_val': list(X_val.shape), 'X_test': list(X_test.shape)},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'sklearn': sklearn.__version__},
}
def _dump(obj, path):
    with open(path, 'w') as fh:
        json.dump(obj, fh, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    print('saved', path)

_dump(project_config, os.path.join(DIRS['config'], 'project_config.json'))
_dump(feature_config, os.path.join(DIRS['config'], 'feature_config.json'))
_dump(report,         os.path.join(DIRS['results'], '01_preprocessing_report.json'))
with open(os.path.join(DIRS['config'], 'mutable_features.txt'), 'w') as fh:
    fh.write('\n'.join(mutable_features))
with open(os.path.join(DIRS['config'], 'fixed_features.txt'), 'w') as fh:
    fh.write('\n'.join(fixed_features))

## 13 · Leakage & integrity checks (reload from Drive)

In [ ]:
P = DIRS['processed']
tr = pd.read_parquet(os.path.join(P, 'train_raw.parquet'))
va = pd.read_parquet(os.path.join(P, 'val_raw.parquet'))
te = pd.read_parquet(os.path.join(P, 'test_raw.parquet'))
fc = json.load(open(os.path.join(DIRS['config'], 'feature_config.json')))
feats = fc['feature_names']

h = {n: set(pd.util.hash_pandas_object(d[feats], index=False).to_numpy()) for n, d in [('train', tr), ('val', va), ('test', te)]}
checks = {
    'train∩test overlapping feature vectors': len(h['train'] & h['test']),
    'train∩val overlapping feature vectors':  len(h['train'] & h['val']),
    'val∩test overlapping feature vectors':   len(h['val'] & h['test']),
    'duplicate rows inside train': int(tr.duplicated(subset=feats).sum()),
    'non-finite values (all splits)': int(sum((~np.isfinite(d[feats].to_numpy())).sum() for d in (tr, va, te))),
}
for k, v in checks.items():
    print(f'{k:<45} {v}')
assert all(v == 0 for v in checks.values()), 'Integrity check failed!'

Xtr = np.load(os.path.join(P, 'X_train.npy')); ytr = np.load(os.path.join(P, 'y_train.npy'))
assert Xtr.shape == (len(tr), len(feats)) and Xtr.dtype == np.float32 and len(ytr) == len(tr)
sc = joblib.load(os.path.join(P, 'scaler.joblib'))
assert np.allclose(sc.transform(tr[feats].to_numpy(np.float32)[:1000]), Xtr[:1000], atol=1e-4)
assert set(fc['mutable_features']).isdisjoint(fc['fixed_features'])
print('\nAll integrity checks passed ✔')
del tr, va, te, Xtr, ytr, h; gc.collect()

## 14 · Summary

In [ ]:
print('=' * 70)
print('STAGE 1 COMPLETE')
print('=' * 70)
print(f'Classes            : {CLASS_NAMES}  (Benign id = {label_to_id[BENIGN_CLASS]})')
print(f'Rows after cleaning: {clean_log["final_rows"]:,}  (from {clean_log["initial_rows"]:,})')
print(f'  removed non-finite {clean_log["removed_nonfinite"]:,} | negative {clean_log["removed_negative"]:,} | '
      f'exact dups {clean_log["removed_exact_duplicates"]:,} | label conflicts {clean_log["removed_label_conflicts"]:,}')
print(f'Features           : {len(FEATURES)}  (mutable {len(mutable_features)}, fixed {len(fixed_features)})')
print(f'Train / Val / Test : {X_train.shape[0]:,} / {X_val.shape[0]:,} / {X_test.shape[0]:,}')
print(pd.DataFrame(split_report['after_balancing']))
print('\nOutputs:')
for d in ('processed', 'config', 'results'):
    for f in sorted(os.listdir(DIRS[d])):
        if d != 'results' or f.startswith('01_'):
            print(f'  {d}/{f}')
print('\nNext: run 02_baseline_models.ipynb')

### Notes for the report (data section)
- Cleaning counts, split sizes and the realistic test mix are in `results/01_preprocessing_report.json`.
- The test set keeps the cleaned dataset's real class proportions, so precision and FPR in notebook 02 reflect a realistic benign-heavy mix; train and validation are balanced.
- De-duplication is done before splitting and label-conflicting flows are removed, so the reported baseline scores are not inflated by near-identical train/test flows (Engelen et al., 2021).
- `mutable_features` / `fixed_features` and the per-class bounds are the contract for Stage 3: the LLM may only change mutable features, every value is clipped to the training bounds, and fixed features are copied unchanged from the source row.